# Modular model building


In [50]:
import distopf as opf
import pyomo.environ as pyo
from distopf.api import create_case
from distopf.pyomo_models import PyoResult, set_objective, solve
from distopf.pyomo_models.common.factory import create_model
from distopf.pyomo_models.network.bfm import BFMProvider
from distopf.pyomo_models.devices.battery import BatteryProvider
from distopf.pyomo_models.devices.capacitor import CapacitorProvider
from distopf.pyomo_models.devices.generator import GeneratorProvider
from distopf.pyomo_models.devices.line_charging import LineChargingProvider
from distopf.pyomo_models.devices.load import LoadProvider
from distopf.pyomo_models.devices.regulator import RegulatorProvider
from distopf.pyomo_models.devices.transformer_shunt import TransformerShuntProvider

First create the model

In [51]:
"""
Load the case into the Case object which contains all of the input data.
Case provides access to the following parameters:
    branch_data: pd.DataFrame,
    bus_data: pd.DataFrame,
    gen_data: Optional[pd.DataFrame] = None,
    cap_data: Optional[pd.DataFrame] = None,
    reg_data: Optional[pd.DataFrame] = None,
    bat_data: Optional[pd.DataFrame] = None,
    schedules: Optional[pd.DataFrame] = None,
    start_step: int = 0,
    n_steps: int = 1,
    delta_t: float = 1,  # hours per step
"""
case = create_case(
    data_path=opf.CASES_DIR / "csv" / "ieee123_30der", start_step=0, n_steps=24
)
# Make any modifications to the case dataframes using Pandas APIs
# Here we ensure that active and reactive power from generators are control variables.
case.gen_data.control_variable = "PQ"

The model is composed from providers. The network core (`BFMProvider`) creates the
bus and branch sets, the power flow variables, the power flow and voltage drop
constraints and the swing bus constraints. Each device provider creates its own
parameters, variables and constraints and adds its injections to the power balance.
Leave a provider out of the list to leave that device out of the model.

In [52]:
core = BFMProvider()
devices = [
    LoadProvider(),  # CVR loads
    GeneratorProvider(),  # P/Q control, circular or octagonal limits
    CapacitorProvider(),
    RegulatorProvider(),
    BatteryProvider(),  # energy, SOC and power limits
    LineChargingProvider(),
    TransformerShuntProvider(),
]
# linear=True selects the LinDistFlow model. Options such as
# circular_constraints=False (octagonal inverter limits) are passed as keywords;
# unknown option names raise a TypeError.
model = create_model(case, core=core, devices=devices, linear=True)

Let us add the following objective function:

$$\min \sum_{t \in \mathcal{T}} \sum_{p \in \phi_j, j:i \rightarrow j} \left(\left(P^{pp}_{ij}(t)\right)^2+\left(Q^{pp}_{ij}(t)\right)^2\right)r^{pp}_{ij}$$

Where:

- $P^{pp}_{ij}(t)$ is the active power flow from bus $i$ to bus $j$ on phase $p$ at time $t$ (`model.p_flow[i, j, p, t]`)
- $Q^{pp}_{ij}(t)$ is the reactive power flow from bus $i$ to bus $j$ on phase $p$ at time $t$ (`model.q_flow[i, j, p, t]`)
- $r^{pp}_{ij}$ is the resistance of branch from bus $i$ to bus $j$ on phase $p$ (`model.r[i, j, p+p]`)
- $p \in \phi_j, j:i \rightarrow j$ represents all phases $p$ and branches from bus $i$ to bus $j$ (`model.branch_phase_set`)
- $\mathcal{T}$ is the set of time steps (`model.time_set`)

In [53]:
def loss_objective(m):
    return pyo.quicksum(
        (m.p_flow[i, j, p, t] ** 2 + m.q_flow[i, j, p, t] ** 2) * m.r[i, j, p + p]
        for i, j, p in m.branch_phase_set
        for t in m.time_set
    )


# set_objective replaces any existing objective. It accepts a rule function like
# this one or a pyo.Objective.
set_objective(model, loss_objective)

Now the model is ready to solve. `solve` runs the solver and returns a `PyoResult` with one dataframe per model variable.

In [ ]:
sol = solve(model, solver="ipopt", duals=False)
print(f"Objective: {sol.objective_value:.4f}, solve time: {sol.solve_time:.1f} s")
t_plot = None  # Time step for plots to show. None animates over all time steps.

In [ ]:
opf.plot_batteries(sol.p_bat, sol.soc)

In [ ]:
opf.plot_voltages(sol.voltages, t=t_plot)

In [ ]:
opf.plot_pq(sol.p_flow, sol.q_flow, t=t_plot)

In [ ]:
opf.plot_polar(sol.p_flow, sol.q_flow, t=t_plot)

In [ ]:
opf.plot_pq(sol.p_gen, sol.q_gen, t=t_plot)

In [ ]:
opf.plot_polar(sol.p_gen, sol.q_gen, t=t_plot)

In [ ]:
opf.plot_network(
    case,
    v=sol.voltages,
    p_flow=sol.p_flow,
    q_flow=sol.q_flow,
    p_gen=sol.p_gen,
    q_gen=sol.q_gen,
    show_reactive_power=True,
    t=t_plot,
)

In [ ]:
# plot_gens(res.p_bat, res.q_bat).show(renderer="browser")
